In [1]:
import glob
from os.path import getctime, join
from utils.files import load_json
import pandas as pd

In [2]:

repeat_experiments = [
    # 'logs/e2e-repeat-Hybrid-EDAC-2025-07-29 00:47:08.777956',
    # 'logs/e2e-repeat-CQL-2025-07-28 19:59:01.108261',
    # 'logs/e2e-repeat-factored-CQL-2025-07-29 10:40:01.758449',
    # 'logs/e2e-repeat-Hybrid-IQL-2025-07-29 12:55:37.044524',
    # 'logs/e2e-repeat-V-CQL-2025-07-29 20:46:42.095383',
    # 'logs/e2e-repeat-V-CQL-2025-07-31 21:47:53.272687',
    'logs/custom_experiments/e2e-extra_eval_discrete_iql-Discrete-IQL-2025-10-07 12:37:46.363482',
    'logs/custom_experiments/e2e-extra_eval_f_cql-F-CQL-2025-11-10 15:08:52.681061'
]

#dataset_fqe_value = -9.43
def load_latest_results(exp_path, results_type):
    result_files = glob.glob(join(exp_path, 'eval', results_type, '**', 'results.json'), recursive=True)
    result_files.sort(key=getctime)
    return load_json(result_files[0])

def flatten(lol):
   for item in lol:
       yield from item
       
def load_experiments_paths(root_path):
    return glob.glob(join(root_path, '*'), recursive=True)

experiments = list(flatten([load_experiments_paths(root_path) for root_path in repeat_experiments]))
experiments

['logs/custom_experiments/e2e-extra_eval_discrete_iql-Discrete-IQL-2025-10-07 12:37:46.363482/0',
 'logs/custom_experiments/e2e-extra_eval_discrete_iql-Discrete-IQL-2025-10-07 12:37:46.363482/2',
 'logs/custom_experiments/e2e-extra_eval_discrete_iql-Discrete-IQL-2025-10-07 12:37:46.363482/4',
 'logs/custom_experiments/e2e-extra_eval_discrete_iql-Discrete-IQL-2025-10-07 12:37:46.363482/1',
 'logs/custom_experiments/e2e-extra_eval_discrete_iql-Discrete-IQL-2025-10-07 12:37:46.363482/3',
 'logs/custom_experiments/e2e-extra_eval_f_cql-F-CQL-2025-11-10 15:08:52.681061/0',
 'logs/custom_experiments/e2e-extra_eval_f_cql-F-CQL-2025-11-10 15:08:52.681061/2',
 'logs/custom_experiments/e2e-extra_eval_f_cql-F-CQL-2025-11-10 15:08:52.681061/4',
 'logs/custom_experiments/e2e-extra_eval_f_cql-F-CQL-2025-11-10 15:08:52.681061/1',
 'logs/custom_experiments/e2e-extra_eval_f_cql-F-CQL-2025-11-10 15:08:52.681061/3']

In [3]:
dfs = []
for experiment in experiments:
    ae_results = load_latest_results(experiment, results_type='Hybrid-Autencoder')
    fqe_results = load_latest_results(experiment, results_type='Dist-fqe')
    config = load_json(join(experiment, 'config.json'))
    algo_name = f"{config['name']}(alpha={config['alpha']})" if config['name'] == 'V-CQL' else config['name']
    print(algo_name)


    experiment_data = [{
        "exp_id": experiment,
        "checkpoint": int(key),
        "algo": algo_name,
        "ood_loss": -val['loss'],
        "fqe_value": fqe_results[key]['value_mean'],
        "root_exp": experiment.split('/')[-3],
    } for key, val in ae_results.items()]
    
    df = pd.DataFrame(experiment_data)
    dfs.append(df)
df = pd.concat(dfs)

Discrete-IQL
Discrete-IQL
Discrete-IQL
Discrete-IQL
Discrete-IQL
F-CQL
F-CQL
F-CQL
F-CQL
F-CQL


In [4]:
df

,exp_id,checkpoint,algo,ood_loss,fqe_value,root_exp
0,logs/custom_experiments/e2e-extra_eval_discret...,100000,Discrete-IQL,-0.394367,-6.792059,custom_experiments
0,logs/custom_experiments/e2e-extra_eval_discret...,100000,Discrete-IQL,-0.401567,-6.941724,custom_experiments
0,logs/custom_experiments/e2e-extra_eval_discret...,100000,Discrete-IQL,-0.405666,-6.952068,custom_experiments
0,logs/custom_experiments/e2e-extra_eval_discret...,100000,Discrete-IQL,-0.397371,-6.879632,custom_experiments
0,logs/custom_experiments/e2e-extra_eval_discret...,100000,Discrete-IQL,-0.398769,-6.833201,custom_experiments
0,logs/custom_experiments/e2e-extra_eval_f_cql-F...,100000,F-CQL,-0.522830,-6.803946,custom_experiments
0,logs/custom_experiments/e2e-extra_eval_f_cql-F...,100000,F-CQL,-0.521950,-7.151465,custom_experiments
0,logs/custom_experiments/e2e-extra_eval_f_cql-F...,100000,F-CQL,-0.509778,-7.218474,custom_experiments
0,logs/custom_experiments/e2e-extra_eval_f_cql-F...,100000,F-CQL,-0.516218,-6.803438,custom_experiments
0,logs/custom_experiments/e2e-extra_eval_f_cql-F...,100000,F-CQL,-0.526111,-7.016297,custom_experiments


In [5]:
checkpoint_map = {
    'V-CQL': 100000,
    'CQL': 100000,
    'factored-CQL': 100000,
    'Hybrid-EDAC': 100000,
    'Hybrid-IQL': 100000,
    'Discrete-IQL': 100000
}

In [6]:

def get_checkpoint_stats(df):
    df = df.copy(deep=True)
    mask = df.apply(
        lambda row: row['checkpoint'] == checkpoint_map.get(row['algo'], 100000),
        axis=1
    )
    filtered = df[mask]
    return filtered

df = df[(df['fqe_value'] <= 0) & (df['fqe_value'] >= -15)]


filtered_df = get_checkpoint_stats(df)
filtered_df

,exp_id,checkpoint,algo,ood_loss,fqe_value,root_exp
0,logs/custom_experiments/e2e-extra_eval_discret...,100000,Discrete-IQL,-0.394367,-6.792059,custom_experiments
0,logs/custom_experiments/e2e-extra_eval_discret...,100000,Discrete-IQL,-0.401567,-6.941724,custom_experiments
0,logs/custom_experiments/e2e-extra_eval_discret...,100000,Discrete-IQL,-0.405666,-6.952068,custom_experiments
0,logs/custom_experiments/e2e-extra_eval_discret...,100000,Discrete-IQL,-0.397371,-6.879632,custom_experiments
0,logs/custom_experiments/e2e-extra_eval_discret...,100000,Discrete-IQL,-0.398769,-6.833201,custom_experiments
0,logs/custom_experiments/e2e-extra_eval_f_cql-F...,100000,F-CQL,-0.522830,-6.803946,custom_experiments
0,logs/custom_experiments/e2e-extra_eval_f_cql-F...,100000,F-CQL,-0.521950,-7.151465,custom_experiments
0,logs/custom_experiments/e2e-extra_eval_f_cql-F...,100000,F-CQL,-0.509778,-7.218474,custom_experiments
0,logs/custom_experiments/e2e-extra_eval_f_cql-F...,100000,F-CQL,-0.516218,-6.803438,custom_experiments
0,logs/custom_experiments/e2e-extra_eval_f_cql-F...,100000,F-CQL,-0.526111,-7.016297,custom_experiments


In [7]:
filtered_df.groupby('algo')[['ood_loss', 'fqe_value']].agg(['mean', 'std'])

ood_loss           fqe_value          
                  mean       std      mean       std
algo                                                
Discrete-IQL -0.399548  0.004292 -6.879737  0.068788
F-CQL        -0.519378  0.006442 -6.998724  0.192356

In [8]:
def pick_tchebycheff(
    df: pd.DataFrame,
    to_minimise: list[str],
    to_maximise: list[str],
    *,
    weights: dict[str, float] | None = None,   # per-objective weights
    targets: dict[str, float] | None = None,   # aspiration point in original units
    rho: float = 1e-6,                         # augmented Tchebycheff tie-breaker
    normalize: bool = True,                    # range-normalize deviations
    score_col: str = "tchebycheff_score",
    return_best: bool = False,
    inplace: bool = False
):
    """
    Adds a Tchebycheff score column (lower is better). Optionally returns the index of the best row.
    - If `targets` is None, uses the best observed values (utopia from data).
    - If `targets` is provided, measures *shortfall* relative to targets (no penalty for exceeding).
    """
    cols = list(dict.fromkeys(to_minimise + to_maximise))
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise KeyError(f"Columns not found in df: {missing}")

    out = df if inplace else df.copy()

    # 1) Orient all objectives so "higher is better"
    sign = {c: (-1.0 if c in to_minimise else 1.0) for c in cols}
    Z = out[cols].copy()
    for c in cols:
        Z[c] = sign[c] * Z[c]

    # 2) Choose the reference (ideal) point in the same oriented space
    if targets is None:
        ideal = Z.max(axis=0)  # best observed after orientation
    else:
        # Transform targets to oriented space (min -> negate, max -> as is)
        t = {}
        for c in cols:
            if c in targets:
                t[c] = sign[c] * targets[c]
        ideal = Z.max(axis=0)  # fallback for any missing targets
        ideal.update(pd.Series(t))

    # 3) Compute normalized shortfall (only penalize falling short of ideal)
    if normalize:
        denom = (Z.max(axis=0) - Z.min(axis=0)).replace(0, 1.0)  # avoid /0
        dev = ((ideal - Z).clip(lower=0)) / denom
    else:
        dev = (ideal - Z).clip(lower=0)

    # 4) Weights (default 1.0 each)
    if weights is None:
        w = pd.Series({c: 1.0 for c in cols}, dtype=float)
    else:
        w = pd.Series({c: float(weights.get(c, 1.0)) for c in cols}, dtype=float)

    wdev = dev.mul(w, axis=1)

    # 5) (Augmented) Tchebycheff score: min over rows of max_i w_i * dev_i  +  rho * sum_i w_i * dev_i
    score = wdev.max(axis=1) + rho * wdev.sum(axis=1)
    out[score_col] = score

    if return_best:
        return out, score.idxmin()
    return out

In [9]:
filtered_df = pick_tchebycheff(filtered_df, to_maximise=['ood_loss', 'fqe_value'], to_minimise=[], targets={'ood_loss': -0.58}, weights={'ood_loss': 2.0})
filtered_df

,exp_id,checkpoint,algo,ood_loss,fqe_value,root_exp,tchebycheff_score
0,logs/custom_experiments/e2e-extra_eval_discret...,100000,Discrete-IQL,-0.394367,-6.792059,custom_experiments,0.000000
0,logs/custom_experiments/e2e-extra_eval_discret...,100000,Discrete-IQL,-0.401567,-6.941724,custom_experiments,0.350983
0,logs/custom_experiments/e2e-extra_eval_discret...,100000,Discrete-IQL,-0.405666,-6.952068,custom_experiments,0.375243
0,logs/custom_experiments/e2e-extra_eval_discret...,100000,Discrete-IQL,-0.397371,-6.879632,custom_experiments,0.205370
0,logs/custom_experiments/e2e-extra_eval_discret...,100000,Discrete-IQL,-0.398769,-6.833201,custom_experiments,0.096484
0,logs/custom_experiments/e2e-extra_eval_f_cql-F...,100000,F-CQL,-0.522830,-6.803946,custom_experiments,0.027876
0,logs/custom_experiments/e2e-extra_eval_f_cql-F...,100000,F-CQL,-0.521950,-7.151465,custom_experiments,0.842856
0,logs/custom_experiments/e2e-extra_eval_f_cql-F...,100000,F-CQL,-0.509778,-7.218474,custom_experiments,1.000001
0,logs/custom_experiments/e2e-extra_eval_f_cql-F...,100000,F-CQL,-0.516218,-6.803438,custom_experiments,0.026685
0,logs/custom_experiments/e2e-extra_eval_f_cql-F...,100000,F-CQL,-0.526111,-7.016297,custom_experiments,0.525868


In [10]:
best_rows = filtered_df.groupby('algo')[['fqe_value', 'ood_loss']].agg(['mean', 'std']).reset_index()
